# Geração de documentação Markdown para RAG — HighBond API

Este notebook analisa o arquivo `openapi.json` (HighBond API Reference) e gera documentação em Markdown com TODA a informação de cada endpoint, pronta para uso em um sistema de RAG.

**O que é gerado:**
1. **Chunks por endpoint** — um `.md` por endpoint em `raw_info/chunks/`. Cada chunk começa com o **resumo do domínio** (ex.: seção "Actions" com a descrição e notas do recurso) seguido da referência completa do endpoint. Inclui `index.md`.
2. **Documento único** — `raw_info/highbond_api_completo.md` com índice + conceitos + todos os endpoints.

**Para cada endpoint**, ambos os formatos incluem: método, path, `operationId`, resumo, descrição completa, e seções **colapsáveis** (`<details>`, todas fechadas por padrão) com:
- **Parâmetros** (path/query) com tipo, obrigatoriedade, enum, default e exemplo — incluindo `$ref` resolvidos;
- **Request body (payload)** com o schema completo em tabela (campos, tipos, obrigatórios, enums, exemplos), com `$ref`/`anyOf`/`oneOf` resolvidos recursivamente;
- **Respostas** por status code, cada uma colapsável, com schema de resposta quando documentado.

Sem classificação de importância — a relevância será definida pelos embeddings no RAG.

In [13]:
import json
import re
from pathlib import Path

SPEC_PATH = Path("openapi.json")
CHUNKS_DIR = Path("raw_info") / "chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = Path("raw_info") / "highbond_api_completo.md"

spec = json.loads(SPEC_PATH.read_text(encoding="utf-8"))

total_ops = sum(
    1
    for methods in spec["paths"].values()
    for op in methods.values()
    if isinstance(op, dict) and "operationId" in op
)
print(f"Spec carregada: {spec['info']['title']} v{spec['info']['version']}")
print(f"Endpoints: {total_ops} | Tags: {len(spec['tags'])} | Schemas: {len(spec['components']['schemas'])}")

Spec carregada: HighBond API Reference v1.1.0
Endpoints: 377 | Tags: 62 | Schemas: 472


## Funções auxiliares
Limpeza de HTML (as descrições do spec contêm tags), resolução de `$ref` e renderização de schemas/parâmetros/payloads/respostas em tabelas Markdown.

In [24]:
def render_schema(schema: dict, payload_rows=None) -> str:
    """Renderiza um schema como tabela markdown de campos (com exemplo se houver).

    payload_rows: linhas já renderizadas do payload do mesmo endpoint — linhas
    idênticas são omitidas da tabela (deduplicação por diferença)."""
    if not isinstance(schema, dict) or not schema:
        return "_Schema não documentado._"
    label = ""
    if "$ref" in schema:
        label = f"**Schema:** `{resolve_ref(schema['$ref'])[0]}`\n\n"
    # Deduplicação: schema de erro padrão renderiza uma linha só
    if "$ref" in schema and resolve_ref(schema["$ref"])[0] == "Errors":
        return label + ERROR_SCHEMA_SUMMARY
    rows = schema_rows(schema)
    if rows:
        omitted = 0
        if payload_rows is not None:
            payload_set = set(payload_rows)
            filtered = [r for r in rows if r not in payload_set]
            omitted = len(rows) - len(filtered)
            rows = filtered
        if not rows:
            return label + "_Mesma estrutura de campos do request body (tabela acima)._"
        lines = ["| Campo | Tipo | Obrigatório | Descrição |", "|-------|------|-------------|-----------|"]
        for path, ptype, req, desc in rows:
            lines.append(f"| `{path}` | {ptype} | {req} | {desc} |")
        note = f"_Apenas campos que diferem do request body. {omitted} campo(s) idênticos omitidos._\n\n" if omitted else ""
        return label + note + "\n".join(lines)
    example = schema.get("example")
    if example is not None:
        return label + "Exemplo:\n\n```json\n" + json.dumps(example, indent=2, ensure_ascii=False)[:2000] + "\n```"
    ptype = schema.get("type")
    return label + (f"_Tipo primitivo: `{ptype}`._" if ptype else "_Schema sem propriedades documentadas._")

## Construção do documento

### Parte 1 — Seções conceituais (introdução da API)
Cada seção `##` da descrição do spec vira uma seção da Parte 1 (auth, regiões, paginação, erros, versionamento etc.).

In [15]:
# Seções conceituais da introdução (auth, paginação, erros, versionamento...)
conceptual_sections = []
intro = clean_html(spec["info"].get("description", ""))
for title, body in split_h2_sections(intro):
    conceptual_sections.append((title, f"## {title}\n\n{body}"))

print(f"Seções conceituais: {len(conceptual_sections)}")
for title, _ in conceptual_sections:
    print(" -", title)

Seções conceituais: 5
 - Introduction
 - Getting started
 - Making requests
 - Errors
 - Versioning


### Parte 2 — Endpoints por domínio, com seções colapsáveis
Para cada tag do spec, agregamos todos os endpoints. Cada endpoint recebe uma âncora própria e seções `<details>` **fechadas** para parâmetros, request body (payload completo) e respostas.

**Cada endpoint também vira um chunk individual**, que começa com o resumo do domínio (ex.: seção "Actions" com a descrição e notas do recurso) seguido da referência completa do endpoint — garantindo contexto em qualquer chunk recuperado pelo RAG.

In [25]:
def render_endpoint(path: str, method: str, op: dict) -> tuple[str, str]:
    """Renderiza um endpoint completo com âncora e <details> fechados. Retorna (anchor, markdown)."""
    anchor = slugify(f"{method.lower()}-{path}-{op['operationId']}")
    lines = [f'<a id="{anchor}"></a>', "", f"### `{method} {path}` — {op.get('summary', '')}", ""]
    lines.append(f"- **operationId:** `{op['operationId']}`")
    lines.append(f"- **Domínio(s):** {', '.join(op.get('tags', []))}")
    desc = clean_html(op.get("description", ""))
    if desc and desc != op.get("summary", ""):
        lines += ["", desc]

    params_md = render_parameters(op)
    if params_md:
        n_params = len(op.get("parameters") or [])
        lines += ["", "<details>", f"<summary><strong>Parâmetros</strong> ({n_params})</summary>", "", params_md, "", "</details>"]

    rb_md, payload_rows = render_request_body(op)
    if rb_md:
        lines += ["", "<details>", "<summary><strong>Request body (payload)</strong></summary>", "", rb_md, "", "</details>"]

    resp_md = render_responses(op, payload_rows=payload_rows)
    if resp_md:
        n_resp = len(op.get("responses") or {})
        lines += ["", "<details>", f"<summary><strong>Respostas</strong> ({n_resp})</summary>", "", resp_md, "", "</details>"]

    return anchor, "\n".join(lines)


def strip_anchor(block: str) -> str:
    """Remove a linha de âncora HTML de um bloco de endpoint."""
    return re.sub(r'^<a id="[^"]+"></a>\n\n', '', block)


# Monta as seções por domínio, um chunk por endpoint e as entradas dos índices
domain_sections = []  # markdown completo por domínio (para o documento único)
endpoint_chunks = []  # {"id", "title", "tags", "content"} — um por endpoint
domain_index_entries = []  # (tag_display, [(anchor, label)]) — índice do documento único
chunk_index_entries = []   # (chunk_id, label) — índice de raw_info/chunks/

for tag in spec["tags"]:
    tag_name = tag["name"]
    endpoints = [(p, m, op) for p, m, op in iter_operations() if tag_name in op.get("tags", [])]
    if not endpoints:
        continue
    display = tag.get("x-displayName", tag_name)
    tag_desc = clean_html(tag.get("description", ""))
    # Resumo do domínio — cabeçalho + descrição, reutilizado no topo de cada chunk de endpoint
    domain_intro = [f"## {display}", ""]
    if tag_desc:
        domain_intro += [tag_desc, ""]

    lines = list(domain_intro)
    entries = []
    for path, method, op in endpoints:
        anchor, block = render_endpoint(path, method, op)
        label = f"`{method} {path}` — {op.get('summary', '')}"
        entries.append((anchor, label))
        lines.append(block)
        lines.append("")

        # Chunk individual por endpoint: resumo do domínio + bloco do endpoint (sem âncora)
        chunk_id = f"endpoint-{anchor}"
        chunk_content = "\n".join(domain_intro + [strip_anchor(block)]).strip()
        endpoint_chunks.append({
            "id": chunk_id,
            "title": f"{display}: {op.get('summary', '')}",
            "tags": ["endpoint", slugify(tag_name)],
            "content": chunk_content,
        })
        chunk_index_entries.append((chunk_id, label))

    domain_sections.append("\n".join(lines))
    domain_index_entries.append((display, entries))

print(f"Chunks de endpoint: {len(endpoint_chunks)}")
print(f"Domínios: {len(domain_sections)}")
print(f"Total de endpoints documentados: {sum(len(e) for _, e in domain_index_entries)}")

Chunks de endpoint: 369
Domínios: 62
Total de endpoints documentados: 369


## Escrita dos arquivos
1. **Chunks por endpoint** em `raw_info/chunks/` — um `.md` por endpoint, começando com o resumo do domínio, com frontmatter YAML (sem importância — definida via embeddings no RAG) + `index.md`.
2. **Documento único** `raw_info/highbond_api_completo.md` — índice com links, Parte 1 (conceitos) e Parte 2 (referência completa).

In [26]:
# --- 1. Chunks por endpoint em raw_info/chunks/ --------------------------------
# Limpa chunks antigos para manter o diretório consistente
for old in CHUNKS_DIR.glob("*.md"):
    old.unlink()

chunk_index_lines = [
    "# Índice de Chunks — HighBond API (RAG)",
    "",
    "Um chunk por endpoint. Cada chunk começa com o resumo do domínio seguido da referência completa do endpoint.",
    "",
    "| Chunk | Endpoint | Tamanho (chars) |",
    "|-------|----------|-----------------|",
]

chunk_sizes_by_id = {}
for c in endpoint_chunks:
    frontmatter = (
        "---\n"
        f"id: {c['id']}\n"
        f"title: \"{c['title'].replace(chr(34), chr(39))}\"\n"
        f"tags: [{', '.join(c['tags'])}]\n"
        f"source: openapi.json (HighBond API v{spec['info']['version']})\n"
        "---\n\n"
    )
    (CHUNKS_DIR / f"{c['id']}.md").write_text(frontmatter + c["content"] + "\n", encoding="utf-8")
    chunk_sizes_by_id[c["id"]] = len(c["content"])

for chunk_id, label in chunk_index_entries:
    chunk_index_lines.append(f"| [{chunk_id}]({chunk_id}.md) | {label} | {chunk_sizes_by_id[chunk_id]} |")

(CHUNKS_DIR / "index.md").write_text("\n".join(chunk_index_lines) + "\n", encoding="utf-8")

# --- 2. Documento único -------------------------------------------------------
doc = []
doc.append(f"# {spec['info']['title']} — Documentação Completa")
doc.append("")
doc.append(f"Versão da API: **{spec['info']['version']}** — gerado a partir de `openapi.json`")
doc.append("")
doc.append("> Todas as seções de parâmetros, payloads e respostas usam blocos `<details>` fechados por padrão.")
doc.append("")
doc.append("## Índice")
doc.append("")
doc.append("### Conceitos fundamentais")
doc.append("")
for title, _ in conceptual_sections:
    doc.append(f"- [{title}](#{slugify(title)})")
doc.append("")
doc.append("### Endpoints por domínio")
doc.append("")
for display, entries in domain_index_entries:
    doc.append(f"**{display}**")
    doc.append("")
    for anchor, label in entries:
        doc.append(f"- [{label}](#{anchor})")
    doc.append("")

doc.append("---")
doc.append("")
doc.append("# Parte 1 — Conceitos fundamentais")
doc.append("")
for _, section in conceptual_sections:
    doc.append(section)
    doc.append("")
doc.append("---")
doc.append("")
doc.append("# Parte 2 — Referência completa de endpoints")
doc.append("")
for section in domain_sections:
    doc.append(section)
    doc.append("")

OUT_FILE.write_text("\n".join(doc) + "\n", encoding="utf-8")

size_kb = OUT_FILE.stat().st_size / 1024
print(f"✅ {len(endpoint_chunks)} chunks + index.md em {CHUNKS_DIR.resolve()}")
print(f"✅ Documento único em {OUT_FILE.resolve()} ({size_kb:.1f} KB)")

✅ 369 chunks + index.md em C:\Users\Rafael Luckner\Documents\repos\uniso\RAG - Contexto para LLMS\rag_doc_api\raw_info\chunks
✅ Documento único em C:\Users\Rafael Luckner\Documents\repos\uniso\RAG - Contexto para LLMS\rag_doc_api\raw_info\highbond_api_completo.md (1996.7 KB)


## Validação
Estatísticas do documento gerado e prévia de um chunk de endpoint, para conferir o resumo do domínio no topo, as tabelas e os blocos colapsáveis.

In [27]:
content = OUT_FILE.read_text(encoding="utf-8")
chunk_sizes = [len(c["content"]) for c in endpoint_chunks]
print(f"Documento único: {len(content)/1024:.1f} KB")
print(f"Chunks de endpoint: {len(chunk_sizes)} | menor: {min(chunk_sizes)} | maior: {max(chunk_sizes)} | média: {sum(chunk_sizes)//len(chunk_sizes)} chars")
print(f"Blocos colapsáveis (<details>): {content.count('<details>')}")
print(f"Âncoras de endpoints: {content.count('<a id=')}")
print(f"Tabelas de payload/schema: {content.count('| Campo | Tipo |')}")
print(f"Tabelas de parâmetros: {content.count('| Nome | Local |')}")

print("\n--- Prévia: chunk do endpoint 'Create an action' (com resumo do domínio no topo) ---\n")
actions_chunk = next((c for c in endpoint_chunks if "createaction" in c["id"]), endpoint_chunks[0])
print(actions_chunk["content"][:2500])

Documento único: 1942.9 KB
Chunks de endpoint: 369 | menor: 862 | maior: 47344 | média: 5438 chars
Blocos colapsáveis (<details>): 1936
Âncoras de endpoints: 369
Tabelas de payload/schema: 460
Tabelas de parâmetros: 369

--- Prévia: chunk do endpoint 'Create an action' (com resumo do domínio no topo) ---

## Actions

An action is a specific follow-up measure that is associated with an identified issue. You can add actions and assign action owners. You can also set up reminders for
yourself to retest issues or track hours spent on retesting by self-assigning actions.

> **Note —**
Only actions in projects that have an `"active"` state can be accessed using the API.

Learn more about actions.

### `POST /orgs/{org_id}/issues/{issue_id}/actions` — Create an action

- **operationId:** `createAction`
- **Domínio(s):** Actions

Create a new action in an existing issue.

Limitations:
* There can be a maximum of 500 actions per issue.
* You cannot create a new action in the following scenarios